# Fixing top-bottom qpath error

In [2]:
import os
import glob
from natsort import natsorted

In [7]:
fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/qu_path/*.qpdata')

In [8]:
fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/qu_path/rep2_Mouse_1_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/qu_path/rep2_Mouse_2_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mouse_3_bot_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mouse_3_top_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/qu_path/rep2_Mouse_4_top_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/qu_path/rep2_Mouse_4_bot_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/qu_path/rep2_Mouse_5_top_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mou

In [9]:
natsorted([fn for fn in fns if ('top' in fn) or ('bot' in fn)])

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/qu_path/Mouse_6_max_proj_bot.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/qu_path/Mouse_6_max_proj_top.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_7/qu_path/Mouse_7_max_proj_bot.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_7/qu_path/Mouse_7_max_proj_top.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_9/qu_path/Mouse_9_max_proj_bot.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_9/qu_path/Mouse_9_max_proj_top.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mouse_3_bot_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mous

In [10]:
fns_to_fix = [fn for fn in fns if ('top' in fn) or ('bot' in fn)]

In [11]:
len(fns_to_fix)

30

In [3]:
from pathlib import Path
import json
from tqdm.auto import tqdm
import numpy as np
import zarr
import dask.array as da
from affine import Affine
from shapely.geometry import shape
from shapely.validation import make_valid
from rasterio import features

In [6]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from shapely.geometry import shape
from shapely.validation import make_valid
from tqdm import tqdm


def count_and_summarize_geojson(geojson_path: Path) -> pd.DataFrame:
    """
    Loads a GeoJSON file, counts the number of valid ROIs (geometries),
    and returns a DataFrame summary of their properties.

    Args:
        geojson_path: Path to the input .geojson file.

    Returns:
        A pandas DataFrame containing the area and is_valid status for each ROI.
    """

    # Define the exclusion threshold in square pixels
    SIZE_THRESHOLD_PX2 = 2000

    print(f"\n🧠 Loading and analyzing {geojson_path.name}")
    try:
        with open(geojson_path) as f:
            gj_data = json.load(f)
    except FileNotFoundError:
        print(f"🛑 Error: File not found at {geojson_path}")
        return pd.DataFrame()

    features = gj_data.get("features", [])
    geoms = []
    properties_list = []
    
    # Initialize counters
    total_features = len(features)
    skipped_large_rois_count = 0
    
    # Iterate through features with a tqdm progress bar
    print(f"Processing GeoJSON features (Exclusion threshold: >{SIZE_THRESHOLD_PX2} px²):")
    for feat in tqdm(features, total=total_features, desc="Validating ROIs"):
        geom = feat.get("geometry")
        properties = feat.get("properties", {})

        if geom:
            g = shape(geom)

            # Skip polygons with area > SIZE_THRESHOLD_PX2 (likely FOV contour)
            if g.area > SIZE_THRESHOLD_PX2:
                skipped_large_rois_count += 1
                continue

            # Fix invalids/self-intersections
            try:
                g = make_valid(g)
            except Exception:
                g = g.buffer(0)

            if not g.is_empty:
                geoms.append(g)
                # Store geometry properties and calculated metrics
                props = {
                    "area_px2": g.area,
                    "is_valid": g.is_valid,
                    "geojson_filename": geojson_path.name,
                }
                # Add QuPath properties if available
                props.update(properties)
                properties_list.append(props)

    print(f"\n--- Analysis Summary ---")
    print(f"Total features in GeoJSON: {total_features}")
    print(f"Count of ROIs skipped due to size (> {SIZE_THRESHOLD_PX2} px²): {skipped_large_rois_count}")
    
    if not geoms:
        print(f"⚠️ No valid ROIs remaining after size filter in {geojson_path.name}")
        return pd.DataFrame()
        
    # Drop the largest polygon (likely the FOV contour) if more than one exists
    # This catches the scenario where the FOV contour was below the SIZE_THRESHOLD_PX2 (unlikely but possible)
    if len(geoms) > 1:
        areas = [g["area_px2"] for g in properties_list]
        max_idx = int(np.argmax(areas))

        # Filter both the geometries and the properties list
        largest_area = areas[max_idx]
        properties_list = [p for i, p in enumerate(properties_list) if i != max_idx]
        geoms = [g for i, g in enumerate(geoms) if i != max_idx]

        # The largest remaining geometry was removed as a final FOV safety check
        if largest_area > SIZE_THRESHOLD_PX2:
             # If the removed largest polygon was above the initial threshold, this is redundant,
             # but we can print a note for clarity.
             pass
        else:
             print(f"Note: The largest remaining ROI (Area: {largest_area:.2f} px²) was removed as a secondary FOV check.")
    
    if not geoms:
        print(f"⚠️ Only potential FOV contour present after final check, skipping.")
        return pd.DataFrame()

    # Create and return the DataFrame summary
    df_summary = pd.DataFrame(properties_list)
    print(f"✅ Found {len(df_summary)} valid ROIs for summary.")
    return df_summary

# --- Execution ---

# Define the single GeoJSON file you want to process
# Replace this with the specific file path you are interested in
geojson_file_to_process = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_12/qu_path/rep2_Mouse_12_bot_max_proj.geojson')

# Generate the DataFrame
results_df = count_and_summarize_geojson(geojson_file_to_process)

# Display the resulting DataFrame
if not results_df.empty:
    print("\nROI Summary DataFrame (First 5 Rows):")
    print(results_df.head())
    print(f"\nTotal ROIs retained in DataFrame: {len(results_df)}")
else:
    print("\nNo DataFrame generated.")


🧠 Loading and analyzing rep2_Mouse_12_bot_max_proj.geojson
Processing GeoJSON features (Exclusion threshold: >2000 px²):


Validating ROIs: 100%|██████████████████████████████████████████████████████| 121092/121092 [00:11<00:00, 10668.66it/s]



--- Analysis Summary ---
Total features in GeoJSON: 121092
Count of ROIs skipped due to size (> 2000 px²): 3027
Note: The largest remaining ROI (Area: 1998.55 px²) was removed as a secondary FOV check.
✅ Found 118064 valid ROIs for summary.

ROI Summary DataFrame (First 5 Rows):
    area_px2  is_valid                    geojson_filename objectType  \
0  111.72460      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
1   34.22840      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
2   82.17700      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
3  180.46655      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
4  116.38620      True  rep2_Mouse_12_bot_max_proj.geojson  detection   

                                        measurements  
0  {'Nucleus: Area': 3.0, 'Nucleus: Perimeter': 1...  
1  {'Nucleus: Area': 1.0, 'Nucleus: Perimeter': 4...  
2  {'Nucleus: Area': 2.3125, 'Nucleus: Perimeter'...  
3  {'Nucleus: Area': 4.9375, 'Nucleus: Perimeter'...  
4  {'

In [8]:
results_df

,area_px2,is_valid,geojson_filename,objectType,measurements
0,111.72460,True,rep2_Mouse_12_bot_max_proj.geojson,detection,"{'Nucleus: Area': 3.0, 'Nucleus: Perimeter': 1..."
1,34.22840,True,rep2_Mouse_12_bot_max_proj.geojson,detection,"{'Nucleus: Area': 1.0, 'Nucleus: Perimeter': 4..."
2,82.17700,True,rep2_Mouse_12_bot_max_proj.geojson,detection,"{'Nucleus: Area': 2.3125, 'Nucleus: Perimeter'..."
3,180.46655,True,rep2_Mouse_12_bot_max_proj.geojson,detection,"{'Nucleus: Area': 4.9375, 'Nucleus: Perimeter'..."
4,116.38620,True,rep2_Mouse_12_bot_max_proj.geojson,detection,"{'Nucleus: Area': 3.125, 'Nucleus: Perimeter':..."
...,...,...,...,...,...
118059,23.00000,True,rep2_Mouse_12_bot_max_proj.geojson,annotation,NaN
118060,44.50000,True,rep2_Mouse_12_bot_max_proj.geojson,annotation,NaN
118061,42.00000,True,rep2_Mouse_12_bot_max_proj.geojson,annotation,NaN
118062,123.00000,True,rep2_Mouse_12_bot_max_proj.geojson,annotation,NaN


# Figuring out top/bottom ness

In [13]:
 list(geojson_dir.glob("*.geojson"))

[PosixPath('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_12/qu_path/rep2_Mouse_12_bot_max_proj.geojson')]

In [11]:
from pathlib import Path
import json
from tqdm.auto import tqdm
import numpy as np
import zarr
import dask.array as da
from affine import Affine
from shapely.geometry import shape
from shapely.validation import make_valid
from rasterio import features

base = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice")

mouse_dir = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_12')
zarr_dir = mouse_dir / "zarr"
geojson_dir = mouse_dir / "qu_path"

# Zarrs to process (skip .prev / .jnotebook)
zarrs = [
    z for z in zarr_dir.glob("*.zarr")
    if not any(s in z.name for s in [".prev", ".jnotebook"])
]

zarr_fn = zarrs[0]

geojsons = list(geojson_dir.glob("*.geojson"))



print(f"\n🧩 Processing {zarr_fn.name}")

# open zarr
root = zarr.open(str(zarr_fn), mode="a")

# use level 0/0 as main image
arr = da.from_zarr(f"{zarr_fn}/0/0")   # (..., Y, X)
shape_yx = arr.shape[-2:]

# identity pixel transform (row, col) -> (y, x)
transform = Affine.identity()

# one merged mask per Zarr
merged_mask = np.zeros(shape_yx, dtype="uint8")




🧩 Processing 20251109_40X_Mouse12b_Timer_PZA-RIF_20251110_11.zarr


In [14]:
arr

dask.array<from-zarr, shape=(1, 3, 17, 54144, 45850), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>

In [51]:
geojsons = list(geojson_dir.glob("*.geojson"))[-1]

In [55]:
geojsons = [geojsons]

In [56]:


for gj in geojsons:
    print(f"  importing {gj.name}")
    with open(gj) as f:
        gj_data = json.load(f)

  importing rep2_Mouse_12_top_max_proj.geojson


In [57]:

for gj in geojsons:
    print(f"  importing {gj.name}")
    with open(gj) as f:
        gj_data = json.load(f)

    geoms = []
    for feat in tqdm(gj_data.get("features", [])):
        geom = feat.get("geometry")
        if geom:
            g = shape(geom)
            if g.area > 2000:  # skip huge polygons (likely FOV contour)
                continue
            # fix invalids/self-intersections
            try:
                g = make_valid(g)
            except Exception:
                g = g.buffer(0)
            if not g.is_empty:
                geoms.append(g)

    if not geoms:
        print("   ⚠️ no geometries found, skipping")
        continue

    # --- drop the largest polygon (likely the FOV contour) ---
    if len(geoms) > 1:
        areas = [g.area for g in geoms]
        max_idx = int(np.argmax(areas))
        geoms = [g for i, g in enumerate(geoms) if i != max_idx]

    if not geoms:
        print("   ⚠️ only FOV contour present, skipping")
        continue
    print('bunnin')
    # burn polygons -> binary mask
    mask = features.rasterize(
        [(g, 1) for g in geoms],
        out_shape=shape_yx,
        fill=0,
        dtype="uint8",
        transform=transform,
    )

    # OR into the merged mask
    merged_mask |= mask

  importing rep2_Mouse_12_top_max_proj.geojson


  0%|          | 0/232292 [00:00<?, ?it/s]

bunnin


In [40]:
# masks_bot = mask
mask_top = mask

In [14]:
import napari
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
import napari

In [30]:
# read the image data
reader = Reader(parse_url('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_12/zarr/20251109_40X_Mouse12b_Timer_PZA-RIF_20251110_11.zarr/0'))
# nodes may include images, labels etc
nodes = list(reader())
# first node will be the image pixel data
image_node = nodes[0]

dask_data = image_node.data


version mismatch: detected: FormatV04, requested: FormatV05


In [32]:
dask_data

[dask.array<from-zarr, shape=(1, 3, 17, 54144, 45850), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(1, 3, 17, 27072, 22925), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(1, 3, 17, 13536, 11462), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(1, 3, 17, 6768, 5731), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(1, 3, 17, 3384, 2865), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(1, 3, 17, 1692, 1432), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(1, 3, 17, 846, 716), dtype=>u2, chunksize=(1, 1, 1, 846, 716), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(1, 3, 17, 423, 358), dtype=>u2, chunksize=(1, 1, 1, 423, 358), chunktype=numpy.ndarray>,
 dask.array<from-z

In [15]:
viewer = napari.Viewer(title = 'testing labels alignment')
# viewer.add_image(dask_data, channel_axis=1)
# viewer.add_labels(mask_top)
# viewer.add_labels(mask_bot)

In [35]:
mask.shape

(54144, 45850)

# Using image to assess shift

In [63]:
tif_fns = glob.glob('/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/rep2/QuPath analysis/Mouse 12/*tif')

In [67]:
tif_fns = [Path(fn) for fn in tif_fns]

In [68]:
import tifffile
from pathlib import Path


shapes = {}

for file_path in tif_fns:
    try:
        # Use TiffFile context manager to open the file
        with tifffile.TiffFile(file_path) as tif:
            # The 'pages' attribute contains the Image File Directories (IFDs),
            # which hold the metadata, including the shape.
            # We assume the main image data shape is found on the first page.
            image_shape = tif.pages[0].shape
            
            # For multi-page/multi-dimensional TIFFs (e.g., time series, Z-stacks),
            # you might need to check the TiffFile 'series' property for the 
            # combined array shape. This is often more reliable for OME-TIFF 
            # or Bio-Formats files.
            # For a simpler case, the shape of the first page is usually sufficient 
            # for the X and Y dimensions.
            
            shapes[file_path.name] = image_shape
            
    except Exception as e:
        shapes[file_path.name] = f"Error reading shape: {e}"

print("Image Shapes:")
for filename, shape in shapes.items():
    print(f"{filename}: {shape}")

# Example output (assuming 3D data: Time, Height, Width):
# Image Shapes:
# first_large_image.tif: (50, 2048, 2048)
# second_large_image.tif: (60, 1024, 1024)

Image Shapes:
rep2_Mouse_12_top_max_proj.tif: (26544, 43196)
rep2_Mouse_12_bot_max_proj.tif: (29409, 39916)


In [70]:
mask.shape

(54144, 45850)

In [72]:
26544 + 29409

55953

In [ ]:


# ---- write as NGFF label: labels/ground_truth_mtb/0 ----
labels_root = root.require_group("labels")
lbl_group = labels_root.require_group("ground_truth_mtb")

# overwrite dataset '0' if it exists
if "0" in lbl_group:
    del lbl_group["0"]

arr_out = lbl_group.create_array(
    "0",
    shape=merged_mask.shape,
    dtype="uint8",
    chunks="auto",
)
arr_out[...] = merged_mask

# minimal NGFF label metadata
lbl_group.attrs["multiscales"] = [{
    "name": "ground_truth_mtb",
    "version": "0.4",
    "axes": [
        {"name": "y", "type": "space", "unit": "pixel"},
        {"name": "x", "type": "space", "unit": "pixel"},
    ],
    "datasets": [{"path": "0"}],
}]
lbl_group.attrs["image-label"] = {
    "version": "0.4",
    "source": "..",   # parent = image root
}

print(f"   ✓ wrote labels/ground_truth_mtb/0 [{merged_mask.shape}]")

print("\n✅ Done — NGFF labels written as labels/ground_truth_mtb/0 in each .zarr")

## Export as 2 separate temporary zarrs

In [8]:
from pathlib import Path
import json
from tqdm.auto import tqdm
import numpy as np
import zarr
import dask.array as da
from affine import Affine
from shapely.geometry import shape
from shapely.validation import make_valid
from rasterio import features
LABEL_KEYWORDS = ["top", "bot", "left", "right"]


In [84]:
og_base = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice")
base_dirs = og_base.glob("rep*")
# Define the keywords and their corresponding NGFF label groups
# The GeoJSON filename must contain one of these keywords to be processed.

LABEL_KEYWORDS = ["top", "bot", "left", "right"]
for base in tqdm(base_dirs, total = 2, desc='iterating over reps'):

    for mouse_dir in tqdm(sorted(base.glob("mouse_*")), total=len(sorted(base.glob("mouse_*")))):
        zarr_dir = mouse_dir / "zarr"
        geojson_dir = mouse_dir / "qu_path"
    
        if not zarr_dir.exists() or not geojson_dir.exists():
            continue
    
        # --- NEW: Filter to only process directories with exactly 2 geojsons ---
        geojsons = list(geojson_dir.glob("*.geojson"))
        if len(geojsons) != 2:
            print(zarr_dir, ' could not find 2 geojson')
            continue
    
        # --- Map GeoJSONs to their intended label based on filename ---
        labeled_geojsons = {}
        for gj in geojsons:
            # Determine the label (top/bot/left/right) from the filename
            label = next((k for k in LABEL_KEYWORDS if k in gj.name), None)
            if label:
                labeled_geojsons[label] = gj
            else:
                # If a geojson doesn't match a required keyword, skip the mouse_dir entirely
                labeled_geojsons = {}
                print(f'No keyword found for geojson {gj}')
                break
        
        # If we didn't find exactly 2 validly named geojsons, skip
        if len(labeled_geojsons) != 2:
            print(f"Skipping {mouse_dir.name}: Found {len(geojsons)} geojsons, but only {len(labeled_geojsons)} matched keywords {LABEL_KEYWORDS}.")
            continue
    
        # Zarrs to process (skip .prev / .jnotebook)
        zarrs = [
            z for z in zarr_dir.glob("*.zarr")
            if not any(s in z.name for s in [".prev", ".jnotebook"])
        ]
        
        if not zarrs:
            continue
    
        for zarr_fn in zarrs:
            print(f"\n🧩 Processing {zarr_fn.name} with 2 GeoJSONs from {mouse_dir.name}")
    
            # open zarr
            # The 'a' mode allows creation and modification of groups/arrays
            root = zarr.open(str(zarr_fn), mode="a")
    
            # use level 0/0 as main image to get shape
            arr = da.from_zarr(f"{zarr_fn}/0/0")  # (..., Y, X)
            shape_yx = arr.shape[-2:]
    
            # identity pixel transform (row, col) -> (y, x)
            transform = Affine.identity()
    
            # --- Process each labeled GeoJSON file ---
            for label, gj in labeled_geojsons.items():
                print(f"  -> Importing mask for label: **{label}** from {gj.name}")
                
                # Reset mask for the current GeoJSON
                current_mask = np.zeros(shape_yx, dtype="uint8")
    
                with open(gj) as f:
                    gj_data = json.load(f)
    
                geoms = []
                for feat in gj_data.get("features", []):
                    geom = feat.get("geometry")
                    if geom:
                        g = shape(geom)
                        if g.area > 2000:  # skip huge polygons (likely FOV contour)
                            continue
                        # fix invalids/self-intersections
                        try:
                            g = make_valid(g)
                        except Exception:
                            g = g.buffer(0)
                        if not g.is_empty:
                            geoms.append(g)
    
                if not geoms:
                    print("    ⚠️ no geometries found for this GeoJSON, skipping")
                    continue
    
                # --- drop the largest polygon (likely the FOV contour) ---
                if len(geoms) > 1:
                    areas = [g.area for g in geoms]
                    max_idx = int(np.argmax(areas))
                    geoms = [g for i, g in enumerate(geoms) if i != max_idx]
    
                if not geoms:
                    print("    ⚠️ only FOV contour present, skipping")
                    continue
    
                # burn polygons -> binary mask
                mask = features.rasterize(
                    [(g, 1) for g in geoms],
                    out_shape=shape_yx,
                    fill=0,
                    dtype="uint8",
                    transform=transform,
                )
                
                # The original code used an OR operation (|=), implying merging multiple geojsons 
                # into one mask. Since we are now processing each geojson to create a separate 
                # mask, we simply assign the result (which is already a merged mask 
                # from polygons *within* this geojson).
                current_mask = mask 
    
                # ---- write as NGFF label: labels/ground_truth_{label}/0 ----
                labels_root = root.require_group("labels")
                # Dynamic label group name: e.g., "ground_truth_top"
                lbl_group_name = f"ground_truth_{label}" 
                lbl_group = labels_root.require_group(lbl_group_name)
    
                # overwrite dataset '0' if it exists
                if "0" in lbl_group:
                    del lbl_group["0"]
    
                arr_out = lbl_group.create_array(
                    "0",
                    shape=current_mask.shape,
                    dtype="uint8",
                    chunks="auto",
                )
                arr_out[...] = current_mask
    
                # minimal NGFF label metadata
                lbl_group.attrs["multiscales"] = [{
                    "name": lbl_group_name,
                    "version": "0.4",
                    "axes": [
                        {"name": "y", "type": "space", "unit": "pixel"},
                        {"name": "x", "type": "space", "unit": "pixel"},
                    ],
                    "datasets": [{"path": "0"}],
                }]
                lbl_group.attrs["image-label"] = {
                    "version": "0.4",
                    "source": "..",  # parent = image root
                }
    
                print(f"    ✓ wrote labels/{lbl_group_name}/0 [{current_mask.shape}]")


print("\n✅ Done — NGFF labels written as labels/ground_truth_{label}/0 in the targeted Zarrs.")

iterating over reps:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr  could not find 2 geojson

🧩 Processing 20251030_40X_TimerMtb_BP_rep2_mice11_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251106_5691.zarr with 2 GeoJSONs from mouse_10
  -> Importing mask for label: **bot** from mouse_10_bot.geojson
    ✓ wrote labels/ground_truth_bot/0 [(64512, 49997)]
  -> Importing mask for label: **top** from mouse_10_top.geojson
    ✓ wrote labels/ground_truth_top/0 [(64512, 49997)]

🧩 Processing 20251030_40X_TimerMtb_BP_rep2_mice11_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251106_5690.zarr with 2 GeoJSONs from mouse_11
  -> Importing mask for label: **bot** from mouse_11_bot.geojson
    ✓ wrote labels/ground_truth_bot/0 [(56218, 45850)]
  -> Importing mask for label: **top** from mouse_11_top.geojson
    ✓ wrote labels/ground_truth_top/0 [(56218, 45850)]

🧩 Processing 20251109_40X_Mouse12b_Timer_PZA-RIF_20251110_11.zarr with 2 GeoJSONs from mouse_12
  -> Import

  0%|          | 0/11 [00:00<?, ?it/s]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_1/zarr  could not find 2 geojson

🧩 Processing 20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr with 2 GeoJSONs from mouse_10
  -> Importing mask for label: **left** from mouse_10_left.geojson
    ✓ wrote labels/ground_truth_left/0 [(41702, 58291)]
  -> Importing mask for label: **right** from mouse_10_right.geojson
    ✓ wrote labels/ground_truth_right/0 [(41702, 58291)]

🧩 Processing 20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.zarr with 2 GeoJSONs from mouse_11
  -> Importing mask for label: **left** from mouse_11_left.geojson
    ✓ wrote labels/ground_truth_left/0 [(41702, 60365)]
  -> Importing mask for label: **right** from mouse_11_right.geojson
    ✓ wrote labels/ground_truth_right/0 [(41702, 60365)]
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_2/zarr  could not find 2 geoj

In [11]:
mouse_dir = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_6')
zarr_dir = mouse_dir / "zarr"
geojson_dir = mouse_dir / "qu_path"

# if not zarr_dir.exists() or not geojson_dir.exists():
    # continue

# --- NEW: Filter to only process directories with exactly 2 geojsons ---
geojsons = list(geojson_dir.glob("*.geojson"))
if len(geojsons) != 2:
    print(zarr_dir, ' could not find 2 geojson')
    # continue

# --- Map GeoJSONs to their intended label based on filename ---
labeled_geojsons = {}
for gj in geojsons:
    # Determine the label (top/bot/left/right) from the filename
    label = next((k for k in LABEL_KEYWORDS if k in gj.name), None)
    if label:
        labeled_geojsons[label] = gj
    else:
        # If a geojson doesn't match a required keyword, skip the mouse_dir entirely
        labeled_geojsons = {}
        print(f'No keyword found for geojson {gj}')
        break

# If we didn't find exactly 2 validly named geojsons, skip
if len(labeled_geojsons) != 2:
    print(f"Skipping {mouse_dir.name}: Found {len(geojsons)} geojsons, but only {len(labeled_geojsons)} matched keywords {LABEL_KEYWORDS}.")
    # continue

# Zarrs to process (skip .prev / .jnotebook)
zarrs = [
    z for z in zarr_dir.glob("*.zarr")
    if not any(s in z.name for s in [".prev", ".jnotebook",])# "top", "bot"])
]

In [12]:
zarrs

[PosixPath('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_6/zarr/rep2_mouse6_bot.zarr'),
 PosixPath('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_6/zarr/rep2_mouse6_top.zarr'),
 PosixPath('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_6/zarr/temp.zarr')]

In [13]:
geojsons

[PosixPath('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_6/qu_path/mouse_6_bot.geojson'),
 PosixPath('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_6/qu_path/mouse_6_top.geojson')]

In [19]:
mouse_dir = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3')
zarr_dir = mouse_dir / "zarr"
geojson_dir = mouse_dir / "qu_path"

# if not zarr_dir.exists() or not geojson_dir.exists():
#     continue

# --- NEW: Filter to only process directories with exactly 2 geojsons ---
geojsons = list(geojson_dir.glob("*.geojson"))
if len(geojsons) != 2:
    print(zarr_dir, ' could not find 2 geojson')
    # continue

# --- Map GeoJSONs to their intended label based on filename ---
labeled_geojsons = {}
for gj in geojsons:
    # Determine the label (top/bot/left/right) from the filename
    label = next((k for k in LABEL_KEYWORDS if k in gj.name), None)
    if label:
        labeled_geojsons[label] = gj
    else:
        # If a geojson doesn't match a required keyword, skip the mouse_dir entirely
        labeled_geojsons = {}
        print(f'No keyword found for geojson {gj}')
        break

# If we didn't find exactly 2 validly named geojsons, skip
if len(labeled_geojsons) != 2:
    print(f"Skipping {mouse_dir.name}: Found {len(geojsons)} geojsons, but only {len(labeled_geojsons)} matched keywords {LABEL_KEYWORDS}.")
    # continue

# Zarrs to process (skip .prev / .jnotebook)
zarrs = [
    z for z in zarr_dir.glob("*.zarr")
    if not any(s in z.name for s in [".prev", ".jnotebook", "top", "bot"])
]

# if not zarrs:
#     continue

for zarr_fn in zarrs:
    print(f"\n🧩 Processing {zarr_fn.name} with 2 GeoJSONs from {mouse_dir.name}")

    # open zarr
    # The 'a' mode allows creation and modification of groups/arrays
    root = zarr.open(str(zarr_fn), mode="a")

    # use level 0/0 as main image to get shape
    arr = da.from_zarr(f"{zarr_fn}/0/0")  # (..., Y, X)
    shape_yx = arr.shape[-2:]

    # identity pixel transform (row, col) -> (y, x)
    transform = Affine.identity()

    # --- Process each labeled GeoJSON file ---
    for label, gj in labeled_geojsons.items():
        print(f"  -> Importing mask for label: **{label}** from {gj.name}")
        
        # Reset mask for the current GeoJSON
        current_mask = np.zeros(shape_yx, dtype="uint8")

        with open(gj) as f:
            gj_data = json.load(f)

        geoms = []
        for feat in gj_data.get("features", []):
            geom = feat.get("geometry")
            if geom:
                g = shape(geom)
                if g.area > 2000:  # skip huge polygons (likely FOV contour)
                    continue
                # fix invalids/self-intersections
                try:
                    g = make_valid(g)
                except Exception:
                    g = g.buffer(0)
                if not g.is_empty:
                    geoms.append(g)

        if not geoms:
            print("    ⚠️ no geometries found for this GeoJSON, skipping")
            continue

        # --- drop the largest polygon (likely the FOV contour) ---
        if len(geoms) > 1:
            areas = [g.area for g in geoms]
            max_idx = int(np.argmax(areas))
            geoms = [g for i, g in enumerate(geoms) if i != max_idx]

        if not geoms:
            print("    ⚠️ only FOV contour present, skipping")
            continue

        # burn polygons -> binary mask
        mask = features.rasterize(
            [(g, 1) for g in geoms],
            out_shape=shape_yx,
            fill=0,
            dtype="uint8",
            transform=transform,
        )
        
        # The original code used an OR operation (|=), implying merging multiple geojsons 
        # into one mask. Since we are now processing each geojson to create a separate 
        # mask, we simply assign the result (which is already a merged mask 
        # from polygons *within* this geojson).
        current_mask = mask 

        # ---- write as NGFF label: labels/ground_truth_{label}/0 ----
        labels_root = root.require_group("labels")
        # Dynamic label group name: e.g., "ground_truth_top"
        lbl_group_name = f"ground_truth_{label}" 
        lbl_group = labels_root.require_group(lbl_group_name)

        # overwrite dataset '0' if it exists
        if "0" in lbl_group:
            del lbl_group["0"]

        arr_out = lbl_group.create_array(
            "0",
            shape=current_mask.shape,
            dtype="uint8",
            chunks="auto",
        )
        arr_out[...] = current_mask

        # minimal NGFF label metadata
        lbl_group.attrs["multiscales"] = [{
            "name": lbl_group_name,
            "version": "0.4",
            "axes": [
                {"name": "y", "type": "space", "unit": "pixel"},
                {"name": "x", "type": "space", "unit": "pixel"},
            ],
            "datasets": [{"path": "0"}],
        }]
        lbl_group.attrs["image-label"] = {
            "version": "0.4",
            "source": "..",  # parent = image root
        }

        print(f"    ✓ wrote labels/{lbl_group_name}/0 [{current_mask.shape}]")


🧩 Processing 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5572.zarr with 2 GeoJSONs from mouse_3
  -> Importing mask for label: **bot** from mouse_3_bot.geojson
    ✓ wrote labels/ground_truth_bot/0 [(43776, 35482)]
  -> Importing mask for label: **top** from mouse_3_top.geojson
    ✓ wrote labels/ground_truth_top/0 [(43776, 35482)]


## and quantify too on OG tif images

In [16]:
import glob
import os
from pathlib import Path
from natsort import natsorted
from tqdm.auto import tqdm
import dask.array as da
from skimage.measure import regionprops
import numpy as np
import pandas as pd

def check_two_geojsons(zarr_path_str):
    """
    Checks if the parent directory of a zarr file has exactly two .geojson files
    in its corresponding 'qu_path' subdirectory.
    """
    # Get the directory containing the zarr file (e.g., mouse_*/zarr)
    zarr_dir = Path(zarr_path_str).parent
    # Get the mouse directory (e.g., mouse_*)
    mouse_dir = zarr_dir.parent
    # Define the GeoJSON directory (e.g., mouse_*/qu_path)
    geojson_dir = mouse_dir / "qu_path"

    if geojson_dir.exists():
        # Count the geojson files
        geojson_count = len(list(geojson_dir.glob("*.geojson")))
        return geojson_count == 2
    return False

# --- Initialization and Filtering ---

zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*.zarr')
zarr_addresses = [fn for fn in zarr_addresses if ('notebook' not in fn) or ('prev' not in fn)]

# Filter Zarr addresses: only keep those whose parent 'mouse_*' dir has 2 geojson files
print(f"Filtering {len(zarr_addresses)} potential Zarr files...")
filtered_zarr_addresses = [
    zarr_addr for zarr_addr in tqdm(zarr_addresses, desc="GeoJSON Check")
    if check_two_geojsons(zarr_addr)
]

zarr_addresses = natsorted(filtered_zarr_addresses)
print(f"Processing {len(zarr_addresses)} Zarr files that meet the 2-GeoJSON criterion.")



Filtering 33 potential Zarr files...


GeoJSON Check:   0%|          | 0/33 [00:00<?, ?it/s]

Processing 20 Zarr files that meet the 2-GeoJSON criterion.


In [17]:
zarr_addresses

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/corrupted_20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_7/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_8/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_8/zarr/corrupted_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr',
 '

In [18]:
from pathlib import Path
import re

def extract_metadata_from_path(mask_zarr_path):
    """
    Extracts the replicate number and mouse number from a full mask Zarr path.
    Assumes the path structure is: .../rep<N>/mouse_<M>/...
    """
    path_obj = Path(mask_zarr_path)
    
    # Convert path to string parts for easier searching
    parts = path_obj.parts
    
    rep_number = None
    mouse_number = None
    
    # Iterate over path parts backwards to find 'rep' and 'mouse' directories efficiently
    for part in reversed(parts):
        if part.startswith('rep') and part[3:].isdigit():
            # Found 'repN', extract N
            rep_number = part[3:]
        elif part.startswith('mouse_'):
            # Found 'mouse_M', extract M
            match = re.search(r'mouse_(\d+)', part)
            if match:
                mouse_number = match.group(1)
        
        # Stop iterating once both are found
        if rep_number is not None and mouse_number is not None:
            break
            
    return rep_number, mouse_number

# Example Usage:
mask_zarr_path = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/labels/ground_truth_bot/0'

rep, mouse_id = extract_metadata_from_path(mask_zarr_path)
rep, mouse_id

('1', '6')

In [119]:
results = []

for zarr_address in tqdm(zarr_addresses, desc="Processing Zarrs"):
    try:
        # Load image data once per Zarr
        # images = da.from_zarr(f"{zarr_address}/0/0")
        
        # --- NEW: Iterate over all possible new label groups ---
        for label in LABEL_KEYWORDS:
            
            mask_zarr_path = f"{zarr_address}/labels/ground_truth_{label}/0"

            
            
            # Check if this specific labeled mask actually exists in the Zarr file
            # Use the zarr library to check existence before trying to load with dask
            try:
                root = zarr.open(zarr_address, mode='r')
                if f"labels/ground_truth_{label}/0" not in root:
                    continue # Skip if this label group/array doesn't exist
            except Exception as e:
                # Handle cases where the zarr path might be inaccessible or malformed
                print(f"Skipping label {label} in {os.path.basename(zarr_address)} due to Zarr access issue: {str(e)}")
                continue

            print(f"  -> Analyzing mask: **{label}**")

            rep, mouse_id = extract_metadata_from_path(mask_zarr_path)
            
            # # Load the specific mask array
            # masks = da.from_zarr(mask_zarr_path)
            
            # # Compute masks once
            # masks_computed = masks.compute()
            
            # load alterante correct images
            if 'rep2' in mask_zarr_path:
                image_path = glob.glob(f'/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/rep{rep}/QuPath analysis/Mouse {mouse_id}/*{label}*.tif')[0]
            else:
                image_path = glob.glob(f'/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_{mouse_id}/QuPath analysis/*{label}*.tif')[0]

            print(mask_zarr_path)
            print(image_path)
            print('')
    except:
        print('something fugged up')

Processing Zarrs:   0%|          | 0/20 [00:00<?, ?it/s]

  -> Analyzing mask: **top**
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/labels/ground_truth_top/0
/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_6/QuPath analysis/Mouse_6_max_proj_top.tif

  -> Analyzing mask: **bot**
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/labels/ground_truth_bot/0
/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_6/QuPath analysis/Mouse_6_max_proj_bot.tif

  -> Analyzing mask: **top**
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/corrupted_20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/labels/ground_truth_top/0
/mnt/NEMO/home/shared/Shared 

# Actual quantification

In [141]:
import os
import zarr
from pathlib import Path
from natsort import natsorted
from tqdm.auto import tqdm
import dask.array as da
from skimage.measure import regionprops, label as cc_label
import numpy as np
import pandas as pd
import re 
from PIL import Image

Image.MAX_IMAGE_PIXELS = 5_000_000_000 


In [126]:
results = []

for zarr_address in tqdm(zarr_addresses, desc="Processing Zarrs"):
    # try:
        # Load image data once per Zarr
        # images = da.from_zarr(f"{zarr_address}/0/0")
        
    # --- NEW: Iterate over all possible new label groups ---
    for label in LABEL_KEYWORDS:
        
        mask_zarr_path = f"{zarr_address}/labels/ground_truth_{label}/0"
        
        # Check if this specific labeled mask actually exists in the Zarr file
        # Use the zarr library to check existence before trying to load with dask
        try:
            root = zarr.open(zarr_address, mode='r')
            if f"labels/ground_truth_{label}/0" not in root:
                continue # Skip if this label group/array doesn't exist
        except Exception as e:
            # Handle cases where the zarr path might be inaccessible or malformed
            print(f"Skipping label {label} in {os.path.basename(zarr_address)} due to Zarr access issue: {str(e)}")
            continue

        print(f"  -> Analyzing mask: **{label}**")

        rep, mouse_id = extract_metadata_from_path(mask_zarr_path)
        
        # Load the specific mask array
        masks = da.from_zarr(mask_zarr_path)
        
        # Compute masks once
        masks_computed = masks.compute()
        labeled_mask = cc_label(masks_computed)
        
        # load alterante correct images
        if 'rep2' in mask_zarr_path:
            image_path = glob.glob(f'/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/rep{rep}/QuPath analysis/Mouse {mouse_id}/*{label}*.tif')[0]
        else:
            image_path = glob.glob(f'/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_{mouse_id}/QuPath analysis/*{label}*.tif')[0]

        with Image.open(image_path) as img:
            # Load with new expanded pix limit
            images = np.array(img)
            print('Loaded image of shape:', images.shape)

        regions = regionprops(labeled_mask) 
    
        region_results = []
    
        for region in tqdm(regions, desc=f"Regions in {label}", leave=True):
            region_id = region.label
        
            # Get bounding box with padding
            pad = 5
            rmin, cmin, rmax, cmax = region.bbox
            rmin = max(0, rmin - pad)
            rmax = min(labeled_mask.shape[0], rmax + pad)
            cmin = max(0, cmin - pad)
            cmax = min(labeled_mask.shape[1], cmax + pad)
        
            # Extract region mask (using the newly labeled mask)
            region_mask_crop = labeled_mask[rmin:rmax, cmin:cmax] == region_id
        
            # Extract image regions (same as before)
            region_ch1 = images[1, rmin:rmax, cmin:cmax]
            region_ch2 = images[2, rmin:rmax, cmin:cmax] # changed due to loading from tif
        
            # Compute max projections for the small region
            max_proj_ch1_region = region_ch1 # region_ch1.max(axis=0).compute()
            max_proj_ch2_region = region_ch2 # .max(axis=0).compute() # already a max proj
        
            # Calculate mean intensity
            mean_intensity_ch1 = np.mean(max_proj_ch1_region[region_mask_crop])
            mean_intensity_ch2 = np.mean(max_proj_ch2_region[region_mask_crop])
        
            # --- MODIFIED: Add mask_label and use full path for zarr_address ---
            region_results.append({
                'zarr_address': mask_zarr_path, # Full path of the mask array
                'rep' : rep,
                'mouse_id' : mouse_id,
                'mask_label': label,           # The mask identifier (top, bot, etc.)
                'region_id': region_id,
                'mean_intensity_ch1': mean_intensity_ch1,
                'mean_intensity_ch2': mean_intensity_ch2,
                'area': region.area,
                'bbox_rmin': rmin,
                'bbox_rmax': rmax,
                'bbox_cmin': cmin,
                'bbox_cmax': cmax
            })
    
        results.extend(region_results)
        results_df.to_pickle(f'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/split_image_fix/rep{rep}_mouse_{mouse_id}_label_{label}_updated.pkl')
    
    # except Exception as e:
    #     print(f"🛑 {os.path.basename(zarr_address)} failed for Zarr file level: {str(e)}")

df_results = pd.DataFrame(results)

Processing Zarrs:   0%|          | 0/20 [00:00<?, ?it/s]

  -> Analyzing mask: **top**


Regions in top:   0%|          | 0/189 [00:00<?, ?it/s]

  -> Analyzing mask: **bot**


TiffFileError: corrupted tag list @773507196

In [128]:
rep, mouse_id

('1', '6')

In [127]:
image_path

'/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_6/QuPath analysis/Mouse_6_max_proj_bot.tif'

In [129]:
os.path.exists(image_path)

True

In [132]:
import cv2

In [134]:
images = cv2.imread(image_path,)# cv2.IMREAD_UNCHANGED)

error: OpenCV(4.12.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:79: error: (-215:Assertion failed) pixels <= CV_IO_MAX_IMAGE_PIXELS in function 'validateInputImageSize'


In [143]:
viewer.add_image(images)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (35221, 44308) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Image layer 'images' at 0x7f678d1fddd0>

In [142]:
images.shape

(35221, 44308)

In [144]:
import dask.array as da
import tifffile as tf

# Use tifffile to read the image lazily. 
# This returns a TiffFile object or a Dask array depending on the configuration.
# The `series=0` argument selects the main image series.
with tf.TiffFile(image_path) as tif:
    # Use asarray() with maxworkers=1 to read into memory/dask array
    # If the image is large, asarray(..., maxworkers=1) can return a Dask array
    # or you can explicitly convert the data structure it returns.
    
    # Check if the data is a single series with multiple channels/bands
    # This assumes a typical multi-channel arrangement: (C, Y, X) or (Y, X, C)
    data = tif.series[0].asarray(out='memmap') 
    
# Convert the resulting data (which might still be large) to a Dask array
# Assuming the data shape is (C, Y, X) or (Y, X, C)
dask_image = da.from_array(data, chunks=(1, 512, 512)) # Adjust chunks for your image dimensions

print(f"Loaded Dask array with shape: {dask_image.shape}") 
# Look for the channel dimension (should be 3) in the shape.

TiffFileError: corrupted tag list @773507196

In [146]:
image_path

'/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_6/QuPath analysis/Mouse_6_max_proj_bot.tif'

In [145]:
import dask.array as da
import tifffile as tf
import numpy as np

# Use the local path here
dask_arrays = []

try:
    with tf.TiffFile(image_path) as tif:
        # Check if the file is truly multi-page/channel (assuming 3 channels = 3 pages)
        num_pages = len(tif.pages)
        print(f"File reports {num_pages} pages.")

        # Iterate over the first three pages/channels, loading each one individually
        for i in range(min(num_pages, 3)):
            # Force tifffile to read the page data directly
            page_data = tif.pages[i].asarray()
            
            # Convert the NumPy array for this page to a Dask array
            darr = da.from_array(page_data, chunks=(512, 512)) # 2D chunks
            dask_arrays.append(darr)
        
    if dask_arrays:
        # Stack the individual Dask arrays along a new channel dimension
        images_dask = da.stack(dask_arrays, axis=0) # Result shape: (C, Y, X)
        print(f"Successfully created Dask array with shape: {images_dask.shape}")
    else:
        print("Failed to load any pages from the TIFF file.")

except Exception as e:
    print(f"🛑 tifffile failed while attempting page-by-page load: {e}")

<tifffile.TiffPages @8> corrupted tag list of page 2 @773507196 raised TiffFileError('suspicious number of tags 4579')
File reports 1 pages.
Successfully created Dask array with shape: (1, 35221, 44308)

<tifffile.TiffPages @8> corrupted tag list of page 2 @773507196 raised TiffFileError('suspicious number of tags 4579')


File reports 1 pages.
Successfully created Dask array with shape: (1, 35221, 44308)


In [150]:
images_dask

dask.array<stack, shape=(1, 35221, 44308), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/tifffile/tifffile.py", line 10206, in _gettags
    raise ValueError(f'suspicious number of tags {tagno}')
ValueError: suspicious number of tags 4579

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1505, in _open_or_raise_error
    added = self._add_layers_with_plugins(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1596, in _add_layers_with_plugins
    layer_data, hookimpl = read_data_with_plugins(
                           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/plugins/io.py", line 78, in read_data_with_plugins
    res = _npe2.read(paths, pl

In [149]:
images = tifffile.imread('/mnt/DATA/mtb_tissue_localisation/Mouse_6_max_proj_bot.tif')

TiffFileError: corrupted tag list @773507196

Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/tifffile/tifffile.py", line 10206, in _gettags
    raise ValueError(f'suspicious number of tags {tagno}')
ValueError: suspicious number of tags 4579

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1505, in _open_or_raise_error
    added = self._add_layers_with_plugins(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1596, in _add_layers_with_plugins
    layer_data, hookimpl = read_data_with_plugins(
                           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/plugins/io.py", line 78, in read_data_with_plugins
    res = _npe2.read(paths, pl

In [5]:
import dask.array as da
import os
from pathlib import Path
import zarr # Used for checking array structure if needed

# Define the path to your main Zarr image directory
# NOTE: Replace 'your_image_path.zarr' with the actual file name QuPath created.
zarr_file_path = Path("/mnt/DATA/mtb_tissue_localisation/mouse_6_bot.ome.zarr")

# The path to the highest resolution array within the Zarr group
# This is typically '0/0' for the first level of the first resolution
NGFF_ARRAY_PATH = "s0" 

try:
    # Load the Zarr array as a Dask array
    # Dask reads the array structure (shape, dtype, chunks) without loading data
    images_dask = da.from_zarr(str(zarr_file_path / NGFF_ARRAY_PATH))
    
    # Check the dimensions
    print(f"\n✅ Successfully loaded Zarr array.")
    print(f"File Path: {zarr_file_path}")
    print(f"Dask Array Shape: {images_dask.shape}")
    print(f"Chunk Layout: {images_dask.chunks}")
    
    # Analyze the dimensions to confirm channel count
    # Assuming shape is (C, Y, X)
    if images_dask.ndim >= 3:
        channels = images_dask.shape[0]
        y_dim = images_dask.shape[-2]
        x_dim = images_dask.shape[-1]
        
        print(f"\nDimensions confirmed:")
        print(f"  Channels (C): {channels}")
        print(f"  Y-dimension: {y_dim}")
        print(f"  X-dimension: {x_dim}")

    # You can now slice and process parts of the image lazily:
    # Example: Accessing the first channel (index 0) and performing a max projection
    # max_projection_ch1 = images_dask[0, :, :].max(axis=0).compute() 

except Exception as e:
    print(f"\n🛑 Error loading Zarr array at {zarr_file_path}: {e}")
    # Provide guidance for checking the structure
    print("\nIf loading fails, verify the exact internal array path (e.g., '0/0', 's0', 'im/0').")


✅ Successfully loaded Zarr array.
File Path: /mnt/DATA/mtb_tissue_localisation/mouse_6_bot.ome.zarr
Dask Array Shape: (3, 35221, 44308)
Chunk Layout: ((1, 1, 1), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 405), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 276))

Dimensions confirmed:
  Channels (C): 3
  Y-dimensio

In [19]:
masks = da.from_zarr(mask_zarr_path)


In [20]:
images_dask

dask.array<from-zarr, shape=(3, 35221, 44308), dtype=>u2, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

In [26]:
for mouse_dir in glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/'):
    os.makedirs(Path(mouse_dir, 'tif'), exist_ok=True)

In [22]:
mask_zarr_path

'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/labels/ground_truth_bot/0'

In [21]:
masks

dask.array<from-zarr, shape=(54144, 45850), dtype=uint8, chunksize=(1692, 1433), chunktype=numpy.ndarray>

In [23]:
viewer.add_labels(masks)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (54144, 45850) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Labels layer 'masks' at 0x78d6272cd650>

In [24]:
viewer.add_image(images_dask, channel_axis=0)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (35221, 44308) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (35221, 44308) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (35221, 44308) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Image' at 0x78d6300f68d0>,
 <Image layer 'Image [1]' at 0x78d8098ad990>,
 <Image layer 'Image [2]' at 0x78d639505650>]

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (54144, 45850) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


# Revisiting loading local tifs

In [30]:
base_dir = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice'
search_pattern = os.path.join(base_dir, 'rep1', 'mouse_*', 'tif', '*.tif')
files = glob.glob(search_pattern)

targets = ['top', 'bot', 'left', 'right']

print(f"Re-scanning {len(files)} files for multidimensional structure...")
print("-" * 60)

for file_path in files:
    file_name = os.path.basename(file_path)

    if not any(sub in file_name for sub in targets):
        continue

    try:
        with tifffile.TiffFile(file_path) as tif:
            # tif.series is the key. It reconstructs the full volume from individual pages.
            # This should report (C, Y, X) or (Y, X, C)
            main_series = tif.series[0]
            dims = main_series.shape
            axes = main_series.axes # Reveals dimension order, e.g., 'CYX'
            
            # We also count the raw pages. If dims is 2D but pages > 1, 
            # the channels exist but aren't automatically grouped by metadata.
            page_count = len(tif.pages)

            print(f"[ANALYSIS] {file_name}")
            print(f"    Series Shape: {dims}")
            print(f"    Axes Layout:  {axes}")
            print(f"    Total Pages:  {page_count}")
            
            # Validation logic: Do we see 3 channels?
            # Check if '3' appears in the shape or if we have 3 raw pages
            if 3 in dims or page_count == 3:
                 print(f"    Status:       CONFIRMED MULTI-CHANNEL")
            else:
                 print(f"    Status:       POTENTIAL MISMATCH (Check dimensions)")

    except Exception as e:
        print(f"[ERROR] {file_name}")
        print(f"    Message: {e}")

    print("-" * 60)

<tifffile.TiffFrame 1 @2617862445> is missing required tags
<tifffile.TiffFrame 1 @526757480> is missing required tags
<tifffile.TiffFrame 1 @2541112979> is missing required tags


Re-scanning 17 files for multidimensional structure...
------------------------------------------------------------
[ERROR] Mouse_11_max_proj_left.tif
    Message: incompatible keyframe
------------------------------------------------------------
[ERROR] Mouse_11_max_proj_right.tif
    Message: incompatible keyframe
------------------------------------------------------------
[ANALYSIS] Mouse_6_max_proj_top.tif
    Series Shape: (3, 24072, 33116)
    Axes Layout:  CYX
    Total Pages:  1
    Status:       CONFIRMED MULTI-CHANNEL
------------------------------------------------------------
[ERROR] Mouse_6_max_proj_bot.tif
    Message: corrupted tag list @773507196
------------------------------------------------------------
[ANALYSIS] Mouse_7_max_proj_top.tif
    Series Shape: (3, 26109, 37250)
    Axes Layout:  CYX
    Total Pages:  1
    Status:       CONFIRMED MULTI-CHANNEL
------------------------------------------------------------
[ERROR] Mouse_7_max_proj_bot.tif
    Message: inco

<tifffile.TiffFrame 1 @3141094959> is missing required tags
<tifffile.TiffFrame 1 @466627624> is missing required tags
<tifffile.TiffFrame 1 @4024859463> is missing required tags


[ANALYSIS] Mouse_8_max_proj_left.tif
    Series Shape: (3, 32229, 30511)
    Axes Layout:  CYX
    Total Pages:  1
    Status:       CONFIRMED MULTI-CHANNEL
------------------------------------------------------------
[ERROR] Mouse_8_max_proj_right.tif
    Message: incompatible keyframe
------------------------------------------------------------
[ANALYSIS] Mouse_9_max_proj_bot.tif
    Series Shape: (3, 23467, 32655)
    Axes Layout:  CYX
    Total Pages:  1
    Status:       CONFIRMED MULTI-CHANNEL
------------------------------------------------------------
[ERROR] Mouse_9_max_proj_top.tif
    Message: incompatible keyframe
------------------------------------------------------------
[ANALYSIS] Mouse_10_max_proj_left.tif
    Series Shape: (3, 35430, 29848)
    Axes Layout:  CYX
    Total Pages:  1
    Status:       CONFIRMED MULTI-CHANNEL
------------------------------------------------------------
[ERROR] Mouse_10_max_proj_right.tif
    Message: incompatible keyframe
---------------

### Moving the rep2 tifs to local

In [37]:
os.path.basename(fn)

'rep2_Mouse_5_bot_max_proj.tif'

In [47]:
import os
import glob
import shutil
import re

# Identify the source matrix
source_pattern = '/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/rep2/QuPath analysis/Mouse */*.tif'
fns_to_copy = [fn for fn in glob.glob(source_pattern) 
               if any(x in fn for x in ['top', 'bot', 'left', 'right'])]

print(f"Targeting {len(fns_to_copy)} files for extraction...")

for fn in tqdm(fns_to_copy):
    try:
        # Ignore temporary constructs
        if 'temp' in fn:
            continue
    
        # Extract the Mouse ID from the path (e.g., 'Mouse 6' -> '6')
        # We look for the pattern "Mouse " followed by digits in the file path
        match = re.search(r'Mouse\s+(\d+)', fn)
        
        if match:
            mouse_N = match.group(1)
        else:
            print(f"[SKIP] Could not identify mouse number for: {fn}")
            continue
    
        basename = os.path.basename(fn)
        
        # Define the new reality
        dest_dir = f'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_{mouse_N}/tif'
        dest = os.path.join(dest_dir, basename)
    
        # Ensure the destination construct exists before copying
        os.makedirs(dest_dir, exist_ok=True)
    
        print(f"Copying: Mouse {mouse_N} | {basename}")
        shutil.copy2(fn, dest)
    except:
        print('error for some reason but continue anyway')

print("Extraction complete.")

Targeting 25 files for extraction...


  0%|          | 0/25 [00:00<?, ?it/s]

Copying: Mouse 5 | rep2_Mouse_5_bot_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 5 | rep2_Mouse_5_top_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 15 | rep2_Mouse_15_top_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 15 | rep2_Mouse_15_bot_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 13 | rep2_Mouse_13_bot_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 13 | rep2_Mouse_13_top_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 6 | rep2_Mouse_6_top_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 6 | rep2_Mouse_6_bot_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 10 | rep2_Mouse_10_bot_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 10 | rep2_Mouse_10_top_max_proj.tif
error for some reason but continue anyway
Copying: Mouse 16 | rep2_Mouse_16_bot_max_proj.tif
error for some reason but c

In [42]:
fn

'/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/rep2/QuPath analysis/Mouse 5/rep2_Mouse_5_bot_max_proj.tif'

In [41]:
dest

'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/tif/rep2_Mouse_5_bot_max_proj.tif'

In [43]:
zarr_image = tifffile.imread(dest, aszarr=True)
dask_image = da.from_zarr(zarr_image)

<tifffile.TiffFrame 1 @2226787671> <TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 330 @2226791549> invalid data type 371')
<tifffile.TiffFrame 1 @2226787671> <TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 324 @2226792917> invalid data type 288')
<tifffile.TiffFrame 1 @2226787671> is missing required tags


RuntimeError: incompatible keyframe

In [44]:
dest

'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/tif/rep2_Mouse_5_bot_max_proj.tif'

In [45]:
images_dask = da.from_zarr('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/zarr/rep2_Mouse_5_bot_max_proj.ome.zarr/s0')


In [46]:
images_dask

dask.array<from-zarr, shape=(3, 30912, 35163), dtype=>u2, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

## Original script

In [ ]:


zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_*/zarr/2025*.zarr')
zarr_addresses = natsorted([fn for fn in zarr_addresses if 'notebook' not in fn])

results = []

for zarr_address in tqdm(zarr_addresses):
    try:
        # Load data
        images = da.from_zarr(f"{zarr_address}/0/0")
        masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")
        
        # Compute masks once
        masks_computed = masks.compute()
        
        # Use regionprops to get bounding boxes efficiently
        regions = regionprops(masks_computed)
        
        region_results = []
        
        for region in tqdm(regions, desc="Processing regions", leave=False):
            region_id = region.label
            
            # Get bounding box with padding
            pad = 5
            rmin, cmin, rmax, cmax = region.bbox
            rmin = max(0, rmin - pad)
            rmax = min(masks_computed.shape[0], rmax + pad)
            cmin = max(0, cmin - pad)
            cmax = min(masks_computed.shape[1], cmax + pad)
            
            # Extract region mask
            region_mask_crop = masks_computed[rmin:rmax, cmin:cmax] == region_id
            
            # Extract image regions
            region_ch1 = images[0, 1, :, rmin:rmax, cmin:cmax]
            region_ch2 = images[0, 2, :, rmin:rmax, cmin:cmax]
            
            # Compute max projections for the small region
            max_proj_ch1_region = region_ch1.max(axis=0).compute()
            max_proj_ch2_region = region_ch2.max(axis=0).compute()
            
            # Calculate mean intensity
            mean_intensity_ch1 = np.mean(max_proj_ch1_region[region_mask_crop])
            mean_intensity_ch2 = np.mean(max_proj_ch2_region[region_mask_crop])
            
            region_results.append({
                'zarr_address': os.path.basename(zarr_address),
                'region_id': region_id,
                'mean_intensity_ch1': mean_intensity_ch1,
                'mean_intensity_ch2': mean_intensity_ch2,
                'area': region.area,
                'bbox_rmin': rmin,
                'bbox_rmax': rmax,
                'bbox_cmin': cmin,
                'bbox_cmax': cmax
            })
        
        results.extend(region_results)
        
    except Exception as e:
        print(f"{zarr_address} failed: {str(e)}")

df_results = pd.DataFrame(results)
